# 4.2 新闻分类：多分类问题示例

本实验使用 Keras 自带的 Reuters 新闻数据集，完成一个典型的**多分类问题**。

## 实验目标

给定一条新闻内容，模型需要判断它属于哪一个新闻类别。

在 Reuters 数据集中，一共有 **46 个类别**，因此这是一个 **46分类问题**。

---

## 本实验将学习到的内容

1. 什么是多分类问题
2. 如何加载 Reuters 新闻数据集
3. 如何把文本数据转换成神经网络能处理的向量
4. 如何构建适用于多分类的神经网络
5. 为什么输出层要用 `softmax`
6. 为什么损失函数要用 `categorical_crossentropy`
7. 如何评估模型分类效果

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.datasets import reuters
from tensorflow.keras import models
from tensorflow.keras import layers

## 1. Reuters 数据集简介

Reuters 是一个新闻数据集，常用于文本分类任务。

在 Keras 中，这个数据集已经被处理好了：
- 每条新闻被表示为一个整数列表
- 每个整数代表一个单词的编号
- 每条新闻对应一个类别标签
- 标签范围一般是 0 到 45，共 46 个类别

为了简化实验，我们只保留训练数据中最常出现的前 10000 个单词。

In [ ]:
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

print("训练集样本数：", len(train_data))
print("测试集样本数：", len(test_data))

In [ ]:
print("第一条训练样本：")
print(train_data[0])

print("\n第一条训练样本对应的类别标签：")
print(train_labels[0])

## 2. 数据格式理解

现在的数据不是原始文本，而是一串整数。

每个整数对应一个单词，一条新闻被转换为一个"单词编号序列"。

神经网络不能直接处理原始字符串，所以这种数字化处理是必须的。

我们可以尝试把这些数字解码回单词看看。

In [ ]:
word_index = reuters.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

# Reuters 数据集中索引同样有偏移，通常需要减去 3
decoded_newswire = " ".join([reverse_word_index.get(i - 3, "?") for i in train_data[0]])
print(decoded_newswire)

## 3. 对输入数据进行向量化

每条新闻的长度不一样，而全连接神经网络要求输入具有固定维度。

这里采用的方法是：
- 建立一个长度为 10000 的向量
- 如果某个单词在新闻中出现过，则该位置记为 1
- 没出现则记为 0

这样，每条新闻最终都被表示为一个 **10000维的0/1向量**

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.
    return results

In [ ]:
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

print("x_train 的形状：", x_train.shape)
print("x_test 的形状：", x_test.shape)

## 4. 对标签进行处理

这是一个 **46分类问题**，所以标签不能只用一个 0 或 1 来表示。

为了让神经网络更好地进行多分类，我们通常把标签转换为 **one-hot 编码**。

### 什么是 one-hot 编码？

假设总共有 46 个类别：
- 如果某条新闻属于第 3 类
- 那么它的标签就表示成一个长度为 46 的向量
- 第 3 个位置为 1，其他位置为 0

例如：`[0, 0, 0, 1, 0, 0, 0, ..., 0]`

In [ ]:
def to_one_hot(labels, dimension=46):
    results = np.zeros((len(labels), dimension))
    for i, label in enumerate(labels):
        results[i, label] = 1.
    return results

In [ ]:
one_hot_train_labels = to_one_hot(train_labels)
one_hot_test_labels = to_one_hot(test_labels)

print("one_hot_train_labels 的形状：", one_hot_train_labels.shape)
print("one_hot_test_labels 的形状：", one_hot_test_labels.shape)

In [ ]:
print("第一条新闻原始标签：", train_labels[0])
print("第一条新闻 one-hot 标签：")
print(one_hot_train_labels[0])

## 5. 构建多分类神经网络

这是一个 46 分类问题，因此输出层必须能够输出 46 个类别各自的概率。

### 网络结构设计
- 输入层：10000维
- 隐藏层1：64个神经元，ReLU
- 隐藏层2：64个神经元，ReLU
- 输出层：46个神经元，Softmax

### 为什么输出层用 softmax？
- 把输出转换成 46 个概率值
- 所有概率之和等于 1

### 为什么损失函数用 categorical_crossentropy？
因为这是一个**多分类问题**，并且标签采用了 **one-hot 编码**。

In [ ]:
model = models.Sequential()
model.add(layers.Dense(64, activation="relu", input_shape=(10000,)))
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(46, activation="softmax"))

In [ ]:
model.compile(
    optimizer="rmsprop",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

## 6. 划分验证集

为了观察模型在未参与训练数据上的表现，我们从训练集中划出一部分作为验证集。

这里取前 1000 条样本作为验证集。

In [ ]:
x_val = x_train[:1000]
partial_x_train = x_train[1000:]

y_val = one_hot_train_labels[:1000]
partial_y_train = one_hot_train_labels[1000:]

print("验证集 x_val 形状：", x_val.shape)
print("训练集 partial_x_train 形状：", partial_x_train.shape)

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val)
)

## 7. 观察训练过程

训练完成后，`history` 中保存了每一轮的：
- 训练损失 `loss`
- 验证损失 `val_loss`
- 训练准确率 `accuracy`
- 验证准确率 `val_accuracy`

In [ ]:
print(history.history.keys())

In [ ]:
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs = range(1, len(loss) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, loss, "bo", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

plt.figure(figsize=(8, 5))
plt.plot(epochs, acc, "bo", label="Training accuracy")
plt.plot(epochs, val_acc, "b", label="Validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 8. 训练结果分析

在这个实验中，通常会出现以下现象：
- 训练损失持续下降
- 验证损失先下降后逐渐上升
- 验证准确率提升到某一程度后趋于稳定甚至下降

这说明模型可能在训练后期出现了**过拟合**。

在这个示例中，常常训练 **8~9轮左右** 比较合适。

In [ ]:
# 重新构建一个新模型
model = models.Sequential()
model.add(layers.Dense(64, activation="relu", input_shape=(10000,)))
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(46, activation="softmax"))

model.compile(
    optimizer="rmsprop",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# 使用较合适的轮数重新训练
model.fit(x_train, one_hot_train_labels, epochs=9, batch_size=512)

## 9. 在测试集上评估模型

测试集用于检验模型在真正未知数据上的表现。

注意：测试集不能参与训练，只能在最终阶段用于评估。

In [ ]:
results = model.evaluate(x_test, one_hot_test_labels)
print("测试损失：", results[0])
print("测试准确率：", results[1])

## 10. 进行预测

模型输出层使用 `softmax`，因此输出结果是一个长度为 46 的概率向量：
- 每个数表示属于某个类别的概率
- 所有概率加起来等于 1
- 概率最大的那个类别，就是模型预测的类别

In [ ]:
predictions = model.predict(x_test)

print("第一条测试新闻的预测概率分布：")
print(predictions[0])

print("\n概率之和：", np.sum(predictions[0]))

In [ ]:
predicted_class = np.argmax(predictions[0])
print("第一条测试新闻预测类别：", predicted_class)
print("第一条测试新闻真实类别：", test_labels[0])

## 11. softmax 输出如何理解？

`softmax` 输出多个概率，适合多分类。例如：
- `[0.01, 0.02, 0.80, 0.03, ...]` 表示第2类概率最大(0.80)

这和二分类中的 `sigmoid` 不同：
- `sigmoid` 输出一个概率，适合二分类
- `softmax` 输出多个概率，适合多分类

## 12. 实验总结

本实验完成了一个典型的**多分类任务：新闻分类**。

### 主要步骤
1. **加载数据**：Reuters 新闻数据集，限制词表 10000 词
2. **输入向量化**：将每条新闻转换成 10000 维向量
3. **标签 one-hot 编码**：将标签转换为长度为 46 的向量
4. **构建模型**：Dense(64, relu) -> Dense(64, relu) -> Dense(46, softmax)
5. **模型编译**：rmsprop + categorical_crossentropy
6. **训练与验证**：观察过拟合现象，选择合适轮数
7. **测试评估**：在独立测试集上验证最终效果

---

### 关键概念
- **多分类问题**：输出类别不止两个
- **one-hot 编码**：把类别标签转换为向量形式
- **softmax**：把输出转换为概率，保证总和为1
- **categorical_crossentropy**：适用于多分类 + one-hot标签
- **过拟合**：训练集表现好但验证集表现差